# 基于指针的C语言的 SIMD 编程

本节我们将学习基于指针的C语言的编程范式，掌握内存申请、同步、搬运、计算等过程的实现方式。并以Softmax算子为例，展示基础指令的使用方式。

本节学习大纲如下：
- 环境准备
- 什么是基于指针的C语言编程
- 内存与指针
- 数据搬运
- Reg矢量计算
- 同步机制
- 算子开发示例
---


## 1. 环境准备

正式开始学习之前，先要对Jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入Jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用bisheng编译器，完成算子的开发及编译。

In [ ]:
import os

!mkdir -p src
import subprocess

result = subprocess.run(
    ['bash', '-l', '-c', 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'],
    capture_output=True, text=True
)
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value
print("\n🎉 Environment initialization process completed successfully!")

## 2. 什么是基于指针的C语言编程
Ascend C为AI Core提供了SIMD编程路径，以及一整套底层核心接口（C API）。通过这套接口能够直接映射NPU硬件指令，提供对向量处理单元、矩阵运算单元、本地存储等硬件资源的精细控制能力。对于追求极致性能、习惯C语言编程的开发者，C语言指针编程是解锁NPU硬件潜能的关键技术路径。
另外，C语言指针编程支持以数组形式分配内存，主要通过指针操作进行编程。提供与业界一致的C语言编程体验。包含asc_simd.h文件来调用C API相应接口。如无特殊说明，包含该头文件即可满足接口调用需求。若API文档中有特殊说明，则应遵循API的具体说明。

```c
#include "c_api/asc_simd.h" 
```

本章旨在帮助开发者建立对C语言指针编程的基础认知，掌握基本开发方法。

## 3. 内存与指针

C语言指针编程通过C风格的地址限定符描述不同层级内存，并可以通过指针直接操作内存地址，从而精准控制数据存放位置。常用存储单元的地址限定符介绍如下：

| 存储单元 | 地址限定符 | 描述 |
|------|-----------|---------|
| Global Memory | \_\_gm\_\_ | 全局内存，输入输出数据 |
| Unified Buffer | \_\_ubuf\_\_ | 片上统一缓冲，计算主存 |
| L1 Buffer | \_\_cbuf\_\_ | L1 缓存，Cube 数据中转 |
| L0A Buffer | \_\_ca\_\_ | Cube 矩阵单元 A 输入 |
| L0B Buffer | \_\_cb\_\_ | Cube 矩阵单元 B 输入 |
| L0C Buffer | \_\_cc\_\_ | Cube 矩阵单元 C 累加输出 |
| Bias Buffer | \_\_biasbuf\_\_ | Bias 专用缓冲区 |

C API支持以数组方式申请片上内存，数组名即为数组首地址，可作为C语言标准语法中的指针使用。例如：

```cpp
constexpr uint32_t ub_size = 256;
__ubuf__ half ub_buffer[ub_size];  // 申请UB内存，32B对齐。

constexpr uint32_t l1_size = 4096;
__cbuf__ half l1_buffer[l1_size];  // 申请L1内存

constexpr uint32_t l0a_size = 128;
__ca__ half l0a_buffer[l0a_size];  // 申请L0A内存，512B对齐。
```

约束：不支持动态数组、多维数组、嵌套数组。

在已申请的数组内存基础上，通过指针偏移获取子地址，可灵活切分内存空间供不同计算阶段复用，开发者需手动确保偏移量满足对应存储空间的对齐要求。

```cpp
constexpr uint32_t ub_size = 1024;
__ubuf__ half ub_buffer[ub_size];  // 申请UB内存，32B对齐。

// 获取UB内存后，用户可自由切分、偏置
__ubuf__ half* tmp_ub_buffer = (__ubuf__ half*)ub_buffer + offset;
```

## 4. 数据搬运

数据搬运与计算是AI Core算子开发的两大核心操作。

Global Memory（以下简称GM）存放输入输出数据，Unified Buffer（以下简称UB）作为矢量计算数据中间缓存。而所有的矢量计算都必须以UB内存作为输入/输出。所以，在计算前，需将数据从GM搬运到UB，计算完成后需要将结果从UB搬运到GM。

Vector搬运接口支撑GM与UB之间的数据搬运，是Vector算子开发的基础。接口命名遵循「源→目标」格式（如`asc_copy_gm2ub`表示GM到UB搬运），参数简洁直观（dst、src、size），主要包含如下接口定义：
- **GM2UB搬运**: `asc_copy_gm2ub`加载输入数据到UB
- **UB2GM搬运**: `asc_copy_ub2gm`输出计算结果到GM

```c
__vector__ __global__ void add_custom(__gm__ float* x_gm, __gm__ float* y_gm, __gm__ float* z_gm)
{
    // 将数据从GM搬运到UB
    asc_copy_gm2ub(x_ub, x_gm + i * tileLength, 1, burst_len, 0, 0);
    // 执行同步及Vector计算
    // 将数据从UB搬运到GM
    asc_copy_ub2gm(z_gm + i * tileLength, z_ub, 1, burst_len, 0, 0);
}
```


## 5. Reg矢量计算

在以往的SIMD矢量编程中，矢量计算单元的源操作数和目的操作数都直接落在UB上。每一条矢量计算的执行过程都包含"从UB读 → 在执行单元算 → 写回UB"。当一个算子由多个矢量计算串联组成时（例如dst = (a + b) * c + d），每一步的中间结果都必须先写回UB，再被下一条指令重新读出。中间结果在UB上往返，访存压力随着矢量计算增加而线性放大。

为解决上述问题，Ascend 950PR/Ascend 950DT新一代架构引入寄存器可编程能力，提供Reg矢量计算接口。其核心思想是：将一段连续的矢量计算保留在寄存器中完成，仅在进入和退出这段计算时与UB交互一次，中间结果可不再回写UB。

### SIMD Register File与执行单元

Reg矢量计算在AIV上执行。AIV是AI Core内用于矢量计算的核，核内参与Reg矢量计算的硬件单元包括：

- **Reg矢量执行单元**：用于执行Reg矢量计算，从寄存器读取数据，完成计算后将结果写回寄存器。
- **DMA单元**：用于执行Reg矢量搬运，负责在寄存器和UB之间搬运数据。
- **Aux Scalar**：处理Reg矢量执行单元和Reg搬运单元所需的标量计算（例如地址计算）。

Reg矢量执行单元、DMA单元和Aux Scalar同属于Reg矢量计算单元，执行时都归属PIPE_V流水。所以，一般将Register/UB间的数据流转当做矢量计算的一部分。

**图1** SIMD Reg矢量执行关系

<img src="./images/Reg执行单元.png" title="Reg执行单元" style="zoom:80%;" />

### 内存层级

AIV的内存层级如下：

**图2** 内存层级关系

<img src="./images/Reg内存层级.png" title="Reg内存层级" style="zoom:100%;" />

在如上的SIMD架构中，UB和寄存器均为每个AIV核内独享的存储空间。参与Vector计算的数据需经过「GM → UB → Register」三级数据流转。 GM存放输入输出数据，UB作为矢量计算数据中间缓存，Register位于Vector计算单元最内层，直接与矢量计算执行单元交互。

C语言指针编程通过直接定义寄存器变量方式进行寄存器编程，根据架构设计，当前包含四类专用寄存器：

| 寄存器名称 | 寄存器类型 | 寄存器描述 | 使用场景 |
|------|-----------|---------|---------|
|矢量数据寄存器 |vector_half、vector_float、vector_int8_t、vector_bfloat16_t等 |用于矢量计算，位宽为VL（Vector Length），可存储VL/sizeof(T)个数据元素。vector_half存储128个half，vector_float存储64个float。 | 矢量计算的基本存储单元，存储计算数据。|
|掩码寄存器 |vector_bool |指示计算过程中哪些元素参与计算，宽度为VL/8。可通过asc_create_mask或asc_update_mask类接口初始化。|控制矢量计算中的元素参与情况，处理尾部数据。 |
|地址寄存器 |iter_reg |存储地址偏移量，循环中根据stride自动自增。通过asc_create_iter_reg类接口初始化。 |循环迭代中自动管理地址偏移，简化地址计算。 |
|非对齐寄存器 |vector_load_unalign、vector_store_unalign |处理起始地址未32字节对齐的数据搬运。用作缓冲区优化UB和寄存器之间不对齐地址访问。需配合asc_loadunalign_pre、asc_storeunalign_post使用。 |UB和寄存器之间连续不对齐地址访问场景。 |

数据寄存器用于存储矢量数据，其位宽为VL（Vector Length），可存储VL/sizeof(T)个数据（T表示数据类型）。在Ascend 950PR/Ascend 950DT版本中， VL = 256B。例如对于矢量数据类型vector_float，该寄存器可存储的元素数量为256B / sizeof(float) = 64个。

```c
uint32_t count = 32;
// 定义并初始化掩码寄存器
vector_bool mask_full = asc_create_mask_b32(PAT_ALL);
vector_bool mask = asc_update_mask_b32(count);

// Defines a register variable of the vector_bool type.
vector_float srcReg;
```
### Reg矢量编程总体结构

基于寄存器（Regbase）的编程模型是在Memory矢量编程的`数据搬入` -> `Compute` -> `数据搬出`编程模型基础上扩展为`数据搬入` -> `Load` -> `Compute`  ->`Store` -> `数据搬出`的Reg矢量计算编程模型。

**图3** Regbase编程总体结构

<img src="./images/Regbase编程模型总体结构.png" title="Regbase编程模型总体结构" style="zoom:100%;" />

### VF函数与执行域

在实现中，`Load` -> `Compute`  ->`Store`的执行过程被定义为Vector Function（以下简称VF函数）。同时还定义了其他两种不同执行域的函数：

| 标签 | 角色 | 调用方 | 可调用对象 |
| --- | --- | --- | --- |
| `__aicore__` | Device侧普通函数 | 核函数或其他`__aicore__`函数 | `__aicore__`函数，或通过`asc_vf_call`调用VF函数 |
| `__simd_vf__` | VF函数 | `__aicore__`函数通过`asc_vf_call`调用 | `__simd_callee__`修饰的Reg矢量API或者VF内部子函数 |
| `__simd_callee__` | VF内部子函数 | `__simd_vf__`或其他`__simd_callee__` | `__simd_callee__` |

这种执行域隔离带来三方面好处：

- **编译期校验**：跨域错误调用（例如在`__aicore__`中直接调用Reg矢量API）会在编译阶段被发现。
- **职责清晰**：`__aicore__`负责GM数据搬运、地址计算等外层逻辑；`__simd_vf__`专注寄存器级计算。
- **优化边界明确**：编译器可以针对VF函数做独立优化（如VF融合），且不影响外层算子结构。

  - **simd vf函数**：上述矢量计算接口只能在simd vf函数中调用， 并且该函数只能通过`asc_vf_call`的方式被核函数/__aicore__函数调用

  ```c
  __simd_vf__ inline void add_vf(__ubuf__ float* dst, __ubuf__ float* src0, __ubuf__ float* src1) {
    // 定义寄存器变量
    vector_float dst_reg, src0_reg, src1_reg;
    // 将数据加载到寄存器变量中
    asc_load(src0_reg, src0);
    // 完成Reg计算
    // 将寄存器变量中的数据保存到UB中
  }
  ```

  - **核函数/__aicore__**: 核函数(使用__global__ __aicore__标识的为核函数)或者普通Device侧函数(使用__aicore__函数)通过`asc_vf_call`函数调用`simd vf`完成最终的计算
  ```c
  __global__ __aicore__ void add_custom(__gm__ float* x, __gm__ float* y, __gm__ float* z)
  {
    __ubuf__ float x_local[TILE_LENGTH];
    __ubuf__ float y_local[TILE_LENGTH];
    __ubuf__ float z_local[TILE_LENGTH];
    // 将数据从GM搬运到UB
    asc_copy_gm2ub_align(x_local, x, BLK_NUM, burst_length, 0, 0, true, 0, src_stride, dst_stride);
    asc_copy_gm2ub_align(y_local, y, BLK_NUM, burst_length, 0, 0, true, 0, src_stride, dst_stride);
    // ...
    // 调用VF函数，完成计算
    asc_vf_call<add_vf>(z_local, x_local, y_local);

    // 将计算结果从UB搬运到GM
    asc_copy_ub2gm_align(z, z_local, BLK_NUM, burst_length, 0, src_stride, dst_stride);
  }
  ```


## 6. 同步机制

AI Core采用多条流水线异步并行架构，数据搬入、计算、数据搬出分别由MTE2、Vector/Cube、MTE3流水线承载。流水线异步执行带来天然并行优势，但也引入数据依赖风险。同步机制是协调流水线协作、确保数据依赖关系得到满足的关键技术。

C语言指针编程同步机制按**同步范围**分为三大类：**核内同步**、**核间同步**、**核间Block内同步**。核内同步解决单个AI Core内部的流水线协作与资源保护问题。核间同步解决多AI Core协作场景的执行进度协调问题。核间Block内同步解决Block内部Vector Core与Cube Core协作问题（Ascend 950PR/950DT专用）。
**表 1**  C语言指针编程同步机制接口概览
|同步范围|同步类型|C指针编程API|适用场景|
|:------|:------|:------|:------|
|核内同步|Mutex（共享资源访问保护）<br><small>📌 Ascend 950PR/950DT引入</small>|<code>asc_lock(pipe, mutex_id)</code><br><code>asc_unlock(pipe, mutex_id)</code>|核内多条流水线访问共享资源（如共享UB区域），避免并发访问导致数据不一致|
|核内同步|SetFlag/WaitFlag（流水线间数据依赖同步）|全局同步：<br><code>asc_sync_notify</code><br><code>asc_sync_wait</code><br>单流水同步：<br><code>asc_sync</code><br><code>asc_sync_pipe</code>|流水线间数据依赖同步，确保搬入完成后计算开始、计算完成后搬出开始|
|核内同步|PipeBarrier（内存访问顺序控制）|<code>asc_sync_data_barrier(arg)|同一流水线内部约束执行顺序，避免"写未完成时读已执行"|
|核间同步|CrossCoreSetFlag/WaitFlag（多AI Core协作）|<code>asc_sync_block_arrive(pipe, flag_id)</code><br><code>asc_sync_block_wait(pipe, flag_id)|多AI Core协作场景，协调多核执行进度|
|核间同步|核间Block内同步<br><small>📌 Ascend 950PR/950DT引入</small>|<code>asc_sync_intra_arrive(pipe, sync_id)</code><br><code>asc_sync_intra_wait(pipe, sync_id)</code><br><small>📌复用CrossCoreSetFlag/WaitFlag接口，mode == 4时使能该能力</small>|Block内Vector Core与Cube Core协作|

本节重点介绍Mutex类型的核内同步。
> 📌 **提示**：Mutex机制仅**Ascend 950PR/950DT**芯片支持，用于核内多流水线共享资源场景。

核内同步机制面向单个AI Core内部，解决流水线间的数据依赖同步、共享资源访问保护、内存访问顺序控制三类问题。

Mutex互斥锁采用**获取锁→访问资源→释放锁**模式，通过互斥访问确保共享资源的正确性。

#### 关键接口说明

- `asc_lock(pipe, mutex_id)`  ：根据mutex_id获取mutex，并阻塞当前流水指令队列，直到对应的mutex_id被unlock。
- `asc_unlock(pipe, mutex_id)`: 直到前置当前流水指令退出后，根据mutex_id释放对应mutex。

```c
uint8_t mutex_id = 1;
asc_lock(PIPE_S, mutex_id);               // Acquire lock
asc_add(shared_buffer, x_local, y_local); // Access shared resource
asc_unlock(PIPE_S, mutex_id);             // Release lock
```


## 7. 算子开发示例

本节将展示Ascend 950PR/950DT产品上，使用Reg矢量计算接口开发的softmax算子的全部过程。本算子将完成一个64 * 128的float类型二维数组的softmax计算。

Softmax的计算公式为：

$$
\text{Softmax}(x_{i,j}) = \frac{e^{x_{i,j} - \max_i}}{\sum_{j=0}^{n-1} e^{x_{i,j} - \max_i}}
$$

本节代码按照如下目录结构存放：

```c
└── src
    ├── softmax.h       // softmax算子Device侧实现
    ├── softmax.asc     // softmax算子Host侧实现
    └── CMakeLists.txt  // CMake配置
```

注意：为了代码组织清晰，Device实现独立放在softmax.h。

### 1. Device侧（NPU设备侧）功能实现

为方便代码管理，Device侧代码实现放在独立的softmax.h文件中。


- VF函数实现

为了更清晰的展示搬运、计算、同步等各类操作之间的关系，本例采用最直接的方式对输入的每行数据依次完成如下操作：

步骤1: 求每行最大值

步骤2: 对最大值做填充扩展

步骤3: 各元素减去最大值（防止溢出）

步骤4: 指数运算

步骤5: 每行求和

步骤6: 求和结果填充扩展

步骤7: 归一化


In [ ]:
%%writefile src/softmax.h

#include "c_api/asc_simd.h"

namespace Custom {
union DataUnion {
    constexpr __aicore__ DataUnion() : f(0.0f) {}
    constexpr __aicore__ DataUnion(uint32_t val) : i(val) {}
    float f;
    uint32_t i;
};
constexpr DataUnion fp32_min_value(0x00800000u);
}

constexpr uint32_t LOOP_COUNT = 1;
constexpr uint32_t WIDTH = 128;
constexpr uint32_t SINGLE_VF_HEIGHT = 64;
constexpr uint32_t SINGLE_VF_DATA_LEN = SINGLE_VF_HEIGHT * WIDTH;
constexpr uint32_t SINGLE_CORE_HEIGHT = SINGLE_VF_HEIGHT * LOOP_COUNT;

__simd_vf__ inline void softmax_vf(__ubuf__ float* dst_ub, __ubuf__ float* src_ub, __ubuf__ float* exp_ub, __ubuf__ float* sum_ub)
{
    constexpr uint16_t one_repeat_cnt = asc_get_vf_len() / sizeof(float);
    uint16_t repeat_times = (WIDTH + one_repeat_cnt - 1) / one_repeat_cnt;
    vector_bool mask_full = asc_create_mask_b32(PAT_ALL);

    vector_bool mask;
    vector_float src_reg;
    vector_float max_reg;
    vector_float exp_reg;
    vector_float sum_reg;
    vector_float div_reg;

    // 第一部分: ReduceMax -> Sub -> Exp
    for (uint16_t i = 0; i < SINGLE_VF_HEIGHT; i++) {
        uint32_t count1 = WIDTH;
        asc_duplicate_scalar(max_reg, Custom::fp32_min_value.f, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count1);
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_max(max_reg, max_reg, src_reg, mask);
        }
        asc_reduce_max(max_reg, max_reg, mask_full);
        asc_duplicate(max_reg, max_reg, mask_full);
        uint32_t count2 = WIDTH;

        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count2);
            asc_loadalign(src_reg, src_ub + i * WIDTH + j * one_repeat_cnt);
            asc_exp_sub(exp_reg, src_reg, max_reg, mask);
            asc_storealign(exp_ub + i * WIDTH + j * one_repeat_cnt, exp_reg, mask);
        }
    }

    // 同步：前置步骤中写入exp_ub的操作完成后才能启动后续步骤。
    asc_mem_bar(VST_VLD);

    // 第二部分: ReduceSum -> Div
    for (uint16_t i = 0; i < SINGLE_VF_HEIGHT; i++) {
        uint32_t count3 = WIDTH;
        asc_duplicate_scalar(sum_reg, 0.0, mask_full);
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count3);
            asc_loadalign(src_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_add(sum_reg, sum_reg, src_reg, mask);
        }
        asc_reduce_sum(sum_reg, sum_reg, mask_full);
        asc_duplicate(sum_reg, sum_reg, mask_full);
        uint32_t count4 = WIDTH;
        for (uint16_t j = 0; j < repeat_times; j++) {
            mask = asc_update_mask_b32(count4);
            asc_loadalign(max_reg, exp_ub + i * WIDTH + j * one_repeat_cnt);
            asc_div(div_reg, max_reg, sum_reg, mask);
            asc_storealign(dst_ub + i * WIDTH + j * one_repeat_cnt, div_reg, mask);
        }
    }
}


- Device侧入口函数：核函数实现

核函数是Device侧提供的，供Host侧调用的入口函数。它接收Host侧提供的__gm__内存指针、Tiling等数据，负责完成如下工作：

a. 结合当前核的ID号，推算出当前核需要处理的数据段。

b. 申请__ub__内存，将本核需要处理的数据搬运到__ub__中。

c. 调用VF函数完成计算。

d. 将计算结果搬出到__gm__内存中本核对应的地址段中。


In [ ]:
%%writefile -a src/softmax.h

// ======================= 核函数 =========================
__global__ __vector__ void softmax_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    asc_init();

    // 申请UB内存
    __ubuf__ float src_ub[64 * 128];
    __ubuf__ float dst_ub[64 * 128];
    __ubuf__ float exp_ub[64 * 128];
    __ubuf__ float sum_ub[64 * 128];

    __gm__ float* x_gm = reinterpret_cast<__gm__ float*>(x) + get_block_idx() * SINGLE_VF_DATA_LEN * LOOP_COUNT;
    __gm__ float* y_gm = reinterpret_cast<__gm__ float*>(y) + get_block_idx() * SINGLE_VF_DATA_LEN * LOOP_COUNT;

    for (uint32_t i = 0; i < LOOP_COUNT; i++) {
        // 上一个循环的数据搬出操作完成后才能启动新的数据搬入操作
        uint8_t mutex_id = 1;
        asc_lock(PIPE_MTE2, mutex_id);
        asc_set_copy_pad_val(float(0.0));
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_gm2ub_align(&src_ub[j * WIDTH], &x_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, true, 0, 0, 0);
        }
        asc_unlock(PIPE_MTE2, mutex_id);

        // 数据搬入操作完成后才能启动计算
        asc_lock(PIPE_V, mutex_id);
        // 调用VF函数执行计算
        asc_vf_call<softmax_vf>(dst_ub, src_ub, exp_ub, sum_ub);
        asc_unlock(PIPE_V, mutex_id);

        // 计算完成后才能启动数据搬出
        asc_lock(PIPE_MTE3, mutex_id);
        for (uint32_t j = 0; j < SINGLE_VF_HEIGHT; j++) {
            asc_copy_ub2gm_align(&y_gm[i * SINGLE_VF_DATA_LEN + j * WIDTH], &dst_ub[j * WIDTH], 1, static_cast<uint16_t>(WIDTH * sizeof(float)), 0, 0, 0);
        }
        asc_unlock(PIPE_MTE3, mutex_id);
    }
}


### 2. Host侧功能实现

Host侧代码是整个算子的入口，运行在CPU上。它负责完成如下功能：

a. 申请Host侧内存空间，并准备输入数据。

b. 准备__gm__内存空间，并将需要计算的数据，拷贝到__gm__内存中。

c. 调用Device侧核函数完成计算。

d. 将计算结果拷贝到Host侧内存中。

e. 校验核函数输出的结果是否符合预期。

为方便代码管理，Host侧代码实现放在独立的softmax.asc中实现。

- 输入数据准备

本段代码在Host侧执行，用于生成指定长度和类型的随机数据，作为softmax算子的输入。

In [ ]:
%%writefile src/softmax.asc

#include <iostream>
#include <vector>
#include <iterator>
#include <random>
#include "acl/acl.h"
#include "softmax.h"

// ======================= 构造随机数据 =========================
int64_t random_key()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_int_distribution<int64_t> dist(0, std::numeric_limits<int64_t>::max());
    return dist(gen);
}

float random_value()
{
    static std::random_device rd;
    static std::mt19937_64 gen(rd());
    static std::uniform_real_distribution<double> dist(0.0, 1.0); // 模板随机 value [0,1]
    return static_cast<float>(dist(gen));
}

void generate_test_data(std::vector<int64_t>& keys, std::vector<float>& values, uint32_t num, uint32_t value_dim)
{
    keys.resize(num);
    values.resize(num * value_dim);

    for (uint32_t i = 0; i < num; ++i) {
        keys[i] = random_key();
        for (uint32_t j = 0; j < value_dim; ++j) {
            values[i * value_dim + j] = random_value();
        }
    }
}

- 生成golden数据

本段代码在Host侧执行。根据输入数据，生成softmax的预期结果，将用于校验softmax算子的输出结果是否正确。

In [ ]:
%%writefile -a src/softmax.asc

// ======================= 构造golden数据 =========================
std::vector<float> softmax_reference(std::vector<float>& src, uint32_t height, uint32_t width)
{
    // 支持 float 和 half 数据类型。half 类型先 cast 为 float 进行计算。
    std::vector<float> dst(src.size());
    for (uint32_t row = 0; row < height; row++) {
        uint32_t offset = row * width;
        // 找最大值
        float max_val = static_cast<float>(src[offset]);
        for (uint32_t col = 1; col < width; col++) {
            max_val = std::max(max_val, static_cast<float>(src[offset + col]));
        }
        // 计算 exp(x - max) 并求和
        float sum_val = 0.0f;
        for (uint32_t col = 0; col < width; col++) {
            float tmp = std::exp(static_cast<float>(src[offset + col]) - max_val);
            dst[offset + col] = static_cast<float>(tmp);
            sum_val += tmp;
        }
        // 归一化
        for (uint32_t col = 0; col < width; col++) {
            dst[offset + col] = static_cast<float>(static_cast<float>(dst[offset + col]) / sum_val);
        }
    }
    return dst;
}

- 结果校验

本段代码在Host侧执行。用于校验softmax算子实际输出的结果和Host侧softmax_reference函数生成golden数据是否相同（float数据类型的误差许可范围一般控制在万分之一）。

In [ ]:
%%writefile -a src/softmax.asc

// ======================= 校验输出结果 =========================
uint32_t verify_vesult(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_func = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };

    print_func(output, "Output");
    print_func(golden, "Golden");
    constexpr float epsilon = 1e-4f;
    bool passed = true;
    uint32_t errCount = 0;
    for (size_t i = 0; i < golden.size(); i++) {
        if (std::abs(static_cast<float>(output[i]) - static_cast<float>(golden[i])) > epsilon) {
            passed = false;
            errCount++;
        }
    }
    if (passed) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed! errCount:" << errCount << std::endl;
        return 1;
    }
}


- Host侧入口函数：main函数实现


In [ ]:
%%writefile -a src/softmax.asc

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t dim = 1;
    constexpr uint32_t height = 64;
    constexpr uint32_t width = 128;
    constexpr uint32_t totalLen = height * width;

    std::vector<int64_t> keys;
    std::vector<float> src(totalLen);
    generate_test_data(keys, src, height, width);
    size_t inputSize = totalLen * sizeof(float);
    uint8_t* src_device = nullptr;
    uint8_t* dst_device = nullptr;
    
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);
    
    aclrtMalloc((void**)&src_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&dst_device, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    
    aclrtMemcpy(src_device, inputSize, src.data(), inputSize, ACL_MEMCPY_HOST_TO_DEVICE);
    
    softmax_custom<<<dim, 0, stream>>>(src_device, dst_device);
    aclrtSynchronizeStream(stream);
    
    std::vector<float> output(totalLen);
    aclrtMemcpy(output.data(), inputSize, dst_device, inputSize, ACL_MEMCPY_DEVICE_TO_HOST);
    
    std::vector<float> golden = softmax_reference(src, height, width);
    uint32_t result = verify_vesult(output, golden);
    
    aclrtFree(src_device);
    aclrtFree(dst_device);
    
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return result;
}


### 3. CMake编译配置

一个完整的算子应用程序需要通过编译生成可执行文件。对于本例中的softmax算子，我们需要配置CMake以支持Ascend C编译工具链，并指定目标架构为dav-3510（Ascend 950PR/Ascend 950DT）。

注意，动态链接库m包含Host侧生成golden数据时所需的数学计算类的方法。

In [ ]:
%%writefile src/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510")

find_package(ASC REQUIRED)
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    softmax.asc
)

set(SOFTMAX_COMMON_LIBS  m)
target_link_libraries(demo PRIVATE ${SOFTMAX_COMMON_LIBS})

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

### 4. 编译和运行

执行以下命令编译可执行文件：

In [ ]:
!cd src && mkdir -p build
!cd src/build/ && \
cmake .. && \
make && \
./demo

## 8. 小结

本章阐述了C语言指针编程的核心定位与技术体系。作为SIMD编程的底层接口，C语言指针编程直接映射NPU硬件指令，提供指令级透明可控能力，帮助开发者实现硬件峰值性能与精准问题定位。编程范式遵循SPMD模型的四步法，核心要点在于显式同步机制的正确应用。


## 课后实践

请编写一个加法算子，完成一维数组的加法运算。

用户需完成如下3个文件：

1、编写add.asc：在该文件中完成Device侧核函数和VF函数的实现。

In [ ]:
%%writefile answer/03.03.04/src/add.asc

2、编写add.h：在该文件中完成Host侧的功能。如测试数据构造、结果校验、核函数调用等。

In [ ]:
%%writefile answer/03.03.04/src/add.h

3、编写CMakeLists.txt：在该文件中完成CMake配置。

In [ ]:
%%writefile answer/03.03.04/src/CMakeLists.txt

执行如下脚本可验证精度是否符合预期：

In [ ]:
!cd answer/03.03.04/src && mkdir -p build
!cd answer/03.03.04/src/build && \
cmake .. && \
make && \
./demo

执行如下命令可查看参考答案：

In [ ]:
!cat answer/03.03.04/add/add.asc

In [ ]:
!cat answer/03.03.04/add/add.h

In [ ]:
!cat answer/03.03.04/add/CMakeLists.txt